# Bounded GAD Demo

This notebook extends the GAD example with **physically bounded** variants and reports side-by-side results for two trajectory sets.

**Motivation.** The raw GAD value (average $\sqrt{\det(\Sigma)}$) is unbounded, which makes direct comparison across scenarios difficult.
We therefore impose a **physical upper bound** based on a maximum plausible speed/step, and map GAD into $[0,1]$ for interpretability.

We provide two bounded mappings: 
1) **Linear normalization**: $\min(GAD / A_{max}, 1)$.
2) **Logistic mapping**: $\sigma(\gamma(GAD - m))$, where $m = A_{max}/2$.

Both preserve ordering while enforcing a bounded scale.


## 1) Load and visualize (same example as before)


In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.mixture import GaussianMixture

# Example PKLs
configs = [
    {
        "input":  "./exampleData/scenario_1_output_imputed.pkl",
        "title":  "Concentrated Predictions",
        "save":   "./bounded_concentrated_kde.png"
    },
    {
        "input":  "./exampleData/scenario_1_output.pkl",
        "title":  "Diverse Predictions",
        "save":   "./bounded_diverse_kde.png"
    }
]

def _to_numpy(pred):
    try:
        import torch
        if isinstance(pred, torch.Tensor):
            return pred.cpu().numpy()
    except ImportError:
        pass
    return np.array(pred)

def _ensure_shape(pred):
    # Accept (N, K, T, D) or (K, T, D); return (N, K, T, D)
    if pred.ndim == 3:
        pred = pred[None, ...]
    return pred

def gad_score(xy, max_k=4):
    """
    xy: shape (N, K, T, 2)
    Returns scalar GAD (paper definition).
    """
    N, K, T, D = xy.shape
    total = 0.0
    for n in range(N):
        for t in range(T):
            data = xy[n, :, t, :]  # (K, 2)
            bics, gmms = [], []
            for k in range(1, max_k + 1):
                gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=0)
                gmm.fit(data)
                bics.append(gmm.bic(data))
                gmms.append(gmm)
            best = gmms[np.argmin(bics)]
            pi, mus, covs = best.weights_, best.means_, best.covariances_

            # Mixture mean
            mu_mix = (pi[:, None] * mus).sum(axis=0)

            # Mixture covariance
            Sigma = np.zeros((D, D))
            for i in range(len(pi)):
                Sigma += pi[i] * covs[i]
                dmu = (mus[i] - mu_mix)[:, None]
                Sigma += pi[i] * (dmu @ dmu.T)

            total += np.sqrt(np.linalg.det(Sigma).real)

    return total / (N * T)

def plot_kde(ax, xy, title):
    pts = xy.reshape(-1, 2).T
    kde = gaussian_kde(pts)
    xmin, ymin = pts.min(axis=1) - 0.1
    xmax, ymax = pts.max(axis=1) + 0.1
    X, Y = np.mgrid[xmin:xmax:200j, ymin:ymax:200j]
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

    flat = Z.ravel()
    levels = np.percentile(flat, [10, 25, 50, 75, 90])

    cf = ax.contourf(X, Y, Z, levels=levels, cmap='Blues', alpha=0.4)
    ax.contour(X, Y, Z, levels=levels, colors='white', linewidths=1)
    ax.set_aspect('equal')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title(title)
    return cf

for cfg in configs:
    with open(cfg["input"], 'rb') as f:
        output = pickle.load(f)
    pred = _to_numpy(output['predicted_trajectory'])

    pred0 = pred[0]
    pred0 = _ensure_shape(pred0)
    xy = pred0[..., :2]

    fig, ax = plt.subplots(figsize=(8, 6))
    cf = plot_kde(ax, xy, f"{cfg['title']}\nKDE Visualization")
    fig.colorbar(cf, ax=ax, label='Density')
    plt.tight_layout()
    fig.savefig(cfg["save"], dpi=300)
    plt.show()
    plt.close(fig)


## 2) Bounded GAD (physical upper bound)

**Assumption.** Let $v_{max}$ be a maximum plausible speed per step (in the same units as the trajectories).
Over $T$ steps, the maximum reachable radius is $L = v_{max} \cdot T$. We use a conservative area bound
$A_{max} = L^2$, consistent with the paper's use of $\sqrt{\det(\Sigma)}$ (which drops the constant $\pi$).

**Linear bounded GAD**: $GAD_{lin} = \min(GAD / A_{max}, 1)$.

**Logistic bounded GAD**: $GAD_{log} = 1/(1 + e^{-\gamma(GAD - m)})$, with $m = A_{max}/2$.

These mappings provide a physically interpretable scale while keeping the relative ordering of diversity values.


In [ ]:
import numpy as np

# Physical parameters (adjust as needed)
V_MAX = 2.0  # max speed per step (same unit as trajectory)

def bound_gad_linear(gad, t_steps, v_max=V_MAX):
    L = v_max * t_steps
    a_max = L ** 2
    return min(gad / a_max, 1.0), a_max

def bound_gad_logistic(gad, t_steps, v_max=V_MAX, gamma=4.0):
    L = v_max * t_steps
    a_max = L ** 2
    m = a_max / 2.0
    return 1 / (1 + np.exp(-gamma * (gad - m))), a_max

results = []

for cfg in configs:
    with open(cfg["input"], 'rb') as f:
        output = pickle.load(f)
    pred = _to_numpy(output['predicted_trajectory'])

    pred0 = pred[0]
    pred0 = _ensure_shape(pred0)
    xy = pred0[..., :2]

    gad = gad_score(xy, max_k=4)
    t_steps = xy.shape[2]

    gad_lin, a_max = bound_gad_linear(gad, t_steps, v_max=V_MAX)
    gad_log, _ = bound_gad_logistic(gad, t_steps, v_max=V_MAX, gamma=4.0)

    results.append({
        'title': cfg['title'],
        'GAD_raw': gad,
        'GAD_linear': gad_lin,
        'GAD_logistic': gad_log,
        'A_max': a_max,
        'T': t_steps,
        'V_MAX': V_MAX,
    })

results


## 3) Comparison Summary


In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df[['title', 'GAD_raw', 'GAD_linear', 'GAD_logistic', 'A_max', 'T', 'V_MAX']]
